In [5]:
#gemini 2.5 flash lite model from google
from langchain_google_genai import ChatGoogleGenerativeAI

# Set your API key (you can also set it via environment variable)
import os
# os.environ["GOOGLE_API_KEY"] = "your-api-key-here"  # Uncomment and add your key

google_api_key = os.environ.get("GOOGLE_API_KEY")


# Initialize the Gemini 2.5 Flash Lite model
chat_model_gemini = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0.7,
    google_api_key=google_api_key
)


In [ ]:
from langchain_google_genai import ChatGoogleEmbeddings

embeddings = ChatGoogleEmbeddings(
    model=" ",
    google_api_key=google_api_key
)

In [6]:
#embeddings model
# Using HuggingFace embeddings (local, no API calls needed)
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

In [7]:
#vector store (in memory)
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

In [8]:
#vector store (chroma)
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db",  # Where to save data locally, remove if not necessary
)


In [16]:
#file loader - load files from local machine
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader, TextLoader
from langchain_community.document_loaders import UnstructuredMarkdownLoader

# Option 1: Load all files from a directory (supports multiple file types)
# Uncomment and modify the path as needed
# loader = DirectoryLoader(
#     path="./data",  # Change this to your directory path
#     glob="**/*.pdf",  # Change pattern to match your file types (e.g., "**/*.md", "**/*.txt")
#     loader_cls=PyPDFLoader,  # Change to TextLoader, UnstructuredMarkdownLoader, etc.
#     show_progress=True,
# )

# Option 2: Load a specific PDF file
loader = PyPDFLoader("./data/WorldEnergyOutlook2025.pdf")

# Option 3: Load a specific markdown file
# loader = UnstructuredMarkdownLoader("./end_product_specification.md")

# Option 4: Load a specific text file
# loader = TextLoader("./info.md", encoding="utf-8")

docs = loader.load()

print(f"Loaded {len(docs)} document(s)")
print(f"Total characters: {sum(len(doc.page_content) for doc in docs)}")

Loaded 519 document(s)
Total characters: 1281133


In [17]:
print(docs[0].page_content[:500])

World Energy
Outlook 
2025


In [18]:
#text splitter - split documents into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=200,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
    
)
all_splits = text_splitter.split_documents(docs)

print(f"Split blog post into {len(all_splits)} sub-documents.")

Split blog post into 1723 sub-documents.


In [19]:
document_ids = vector_store.add_documents(documents=all_splits)

print(document_ids[:3])

['9109560a-ac3d-4f02-8ca6-1ba8047cb1f6', 'bd95eec4-7a6a-4860-97b0-d104de56ec46', 'da8b8073-0c56-4e2b-8b79-96ef75bad4a1']


In [ ]:
# ============================================================================
# AGENT CREATION - PART 1: Define Tools
# ============================================================================
# Tools are functions that the agent can call to perform actions.
# The agent will decide when and how to use these tools based on the user's query.
# This is a key component of the agent architecture.

from langchain.tools import tool

# Define a tool for retrieving context from the vector store
# The @tool decorator converts this function into a LangChain tool that the agent can use
@tool
def retrieve_context(query: str):
    """
    Retrieve information to help answer a query.
    
    This tool performs semantic search in the vector store to find relevant documents.
    The agent will call this tool when it needs to search for information to answer questions.
    """
    # Search the vector store for the k=3 most similar documents to the query
    retrieved_docs = vector_store.similarity_search(query, k=3)
    
    # Format the retrieved documents into a readable string
    # This serialized text will be passed to the LLM as context
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    
    # Return the serialized text (for LLM to process)
    return serialized


In [ ]:
# ============================================================================
# AGENT CREATION - PART 2: Create the Agent
# ============================================================================
# This is where we assemble all components to create the LangChain agent.
# The agent combines:
#   1. A language model (chat_model_gemini) - for reasoning and text generation
#   2. Tools (retrieve_context) - for performing actions like searching
#   3. A system prompt - to guide the agent's behavior

from langchain.agents import create_agent

# Step 1: Define the list of tools the agent can use
# The agent will automatically decide when to call these tools based on the query
tools = [retrieve_context]

# Step 2: Define the system prompt
# This prompt instructs the agent on how to behave and when to use tools
# The explicit instruction to use retrieve_context ensures the agent performs RAG
prompt = (
    "You are a helpful assistant that answers questions using information from a book. "
    "IMPORTANT: You MUST use the retrieve_context tool to search for relevant information "
    "before answering any question. Do not answer without first using the tool. "
    "After retrieving context, provide a clear and accurate answer based on the retrieved information."
)

# Step 3: Create the agent
# create_agent() combines the model, tools, and prompt into an agent executor
# The agent can now:
#   - Understand user queries
#   - Decide when to use tools
#   - Call retrieve_context to search the vector store
#   - Generate answers based on retrieved context
agent = create_agent(chat_model_gemini, tools, system_prompt=prompt)

In [ ]:
# ============================================================================
# AGENT CREATION - PART 3: Execute the Agent
# ============================================================================
# This demonstrates how to use the created agent to answer questions.
# The agent will:
#   1. Receive the user query
#   2. Decide to use the retrieve_context tool
#   3. Call the tool to search the vector store
#   4. Receive the retrieved context
#   5. Generate an answer based on the context

# Define the user's question
query = (
    """
    Document(metadata={'source': './data/bitcoin.jpeg'}, page_content='Bitcoin Price at New Record High Daily Bitcoin price in U.S. dollars\n\n(2017-2021)"\n\n\n\n35,000 Jan 03, 2021 30,000 32,782 25,000 Dec 16, 2017 19,497 20,000 15,000 10,000 5,000 0 2017 2018 2019 2020\n\n‘21\n\n* Closing price (latest data in range, UTC time) Source: Coin Market Cap\n\nOOO\n\nStatista %')
    """
)

# Execute the agent with streaming
# agent.stream() allows us to see the agent's reasoning process in real-time
# stream_mode="values" returns the full state at each step
for event in agent.stream(
    {"messages": [{"role": "user", "content": query}]},  # Format: list of messages
    stream_mode="values",  # Return complete state at each step
):
    # Pretty print the last message in each event to see:
    # - Human messages (user queries)
    # - Tool calls (when agent decides to use retrieve_context)
    # - Tool results (retrieved documents)
    # - AI messages (final answers)
    event["messages"][-1].pretty_print()

================================ Human Message =================================


    Document(metadata={'source': './data/bitcoin.jpeg'}, page_content='Bitcoin Price at New Record High Daily Bitcoin price in U.S. dollars

(2017-2021)"



35,000 Jan 03, 2021 30,000 32,782 25,000 Dec 16, 2017 19,497 20,000 15,000 10,000 5,000 0 2017 2018 2019 2020

‘21

* Closing price (latest data in range, UTC time) Source: Coin Market Cap

OOO

Statista %')
    
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (f92be6ac-29d2-4094-9ce1-ac44f1a43116)
 Call ID: f92be6ac-29d2-4094-9ce1-ac44f1a43116
  Args:
    query: Bitcoin price history from 2017 to 2021
================================= Tool Message =================================
Name: retrieve_context

Source: {'creationdate': '2025-11-11T18:43:06+01:00', 'title': 'World Energy Outlook 2025', 'author': 'International Energy Agency', 'page_label': '498', 'producer': 'Adobe Acrobat (64-

In [32]:
#from langchain_community.document_loaders.image import UnstructuredImageLoader

#  loader = UnstructuredImageLoader("./data/bitcoin.jpeg")

#data = loader.load()

#data[0]

In [33]:
vector_store.delete(
    where={"source": "./data/WorldEnergyOutlook2025.pdf"}
)